# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local agentic pipeline for English-to-Chinese cross-lingual dialogue summarization.

The pipeline uses three local small language model agents. Agent 1 first extracts structured information from the original English dialogue. Agent 2 then selects the most important information and creates summary points from the structured representation. Agent 3 generates the final Chinese summary from the summary points and verifies whether the output is complete and non-hallucinatory.

```text
English Dialogue
→ Agent 1: Information Extraction Agent
→ Agent 2: Summary Planning Agent
→ Agent 3: Translation-Verification Agent
→ Final Chinese Summary

The pipeline consists of three agents:

```text
Agent 1: Information Extraction Agent
Input: original English dialogue
Output: structured representation of the dialogue

Agent 2: Summary Planning Agent
Input: structured representation from Agent 1
Output: summary points / summary plan

Agent 3: Translation-Verification Agent
Input: summary points from Agent 2
Output: final Chinese summary with verification
```
The local small language models are served through Ollama. The notebook controls the agent workflow, prompt design, input/output processing, JSON parsing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [3]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Agent models
SEMANTIC_UNDERSTANDING_MODEL = "qwen3.5:9b"
SUMMARY_GENERATION_MODEL = "qwen3.5:9b"
TRANSLATION_MODEL = "translategemma:12b"

# If translategemma:12b is too slow, use:
# TRANSLATION_MODEL = "translategemma:4b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Input dataset path
RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
FULL_OUTPUT_PATH = OUTPUT_DIR / "1_semantic_summary_translation_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "1_semantic_summary_translation_final_summaries.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "1_semantic_summary_translation_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_outputs.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_errors.jsonl


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: Prompt templates

# Agent 1: Linguistic Robustness & Disambiguation
SEMANTIC_UNDERSTANDING_PROMPT = """Analyze this English dialogue with a focus on 'Pragmatic Competence' and 'Disambiguation'.

Extract:
- Participants: List speakers.
- Illocutionary Acts: Identify the actual intent behind speech (e.g., Suggestion vs. Command, Indirect Refusal).
- Semantic Roles: Define 'Who did what to whom' clearly to ensure linguistic robustness.
- Contextual Evidence: Short quotes that justify your interpretation of intent.

Focus: Resolve ambiguity in colloquialisms (e.g., "Just text him" as a refusal of a request).

Output schema:
{
  "participants": ["string"],
  "semantic_grounding": [
    {
      "speaker": "string",
      "speech_act": "string",
      "intended_meaning": "string",
      "evidence": "string"
    }
  ]
}

Input dialogue:
{dialogue}

JSON output:
"""


# Agent 2: Resolution & Intent Planning (Updated to align with Agent 1's semantic_grounding schema)
SUMMARY_GENERATION_PROMPT = """Select the most important points from the structured semantic representation.

Return JSON with:
summary_points: most important resolved intents and agreements

Guidelines:
- Focus on 'Final Agreements' or 'Resolved Actions' based on the 'intended_meaning' and 'speech_act'.
- If a request was refused, the refusal is the core event.
- Select only 1-3 points.

Output schema:
{
  "summary_points": ["string"]
}

Input structured semantic representation:
{semantic_representation}

JSON output:
"""


# Agent 3: Cultural Nuance & Pragmatic Translation
TRANSLATION_PROMPT = """Generate a Chinese summary that captures the 'Cultural Nuance' and 'Register' of the dialogue.

Guidelines:
- Avoid literal translation; use natural Chinese idiomatic expressions.
- Capture the interpersonal dynamics (e.g., politeness, urgency, or casualness) appropriate for a Chinese context.
- Verification: Ensure the 'Social Intent' of the original speakers is preserved in the Chinese output.

Output schema:
{
  "summary_zh": ["string"],
  "cultural_verification": {
    "nuance_preserved": true,
    "naturalness_score": "1-5"
  }
}

Input English summary points:
{summary_points}

JSON output:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: Agent functions

import re

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace only named placeholders while keeping JSON braces in the prompt unchanged."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def extract_json_object(text: str) -> Dict[str, Any]:
    """Extract and parse a JSON object from a model response.

    This function is intentionally robust because small local models may:
    - wrap JSON in markdown fences,
    - add extra text before or after JSON,
    - output Python-style booleans such as True/False.
    """
    raw = text.strip()

    # Remove markdown fences if present.
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(
            line for line in lines
            if not line.strip().startswith("```")
        ).strip()

    candidates = [raw]

    # Try extracting the substring between the first "{" and the last "}".
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidates.append(raw[start:end + 1])

    for candidate in candidates:
        # First try direct JSON parsing.
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

        # Repair common Python-style JSON issues.
        repaired = candidate
        repaired = re.sub(r"\bTrue\b", "true", repaired)
        repaired = re.sub(r"\bFalse\b", "false", repaired)
        repaired = re.sub(r"\bNone\b", "null", repaired)

        try:
            parsed = json.loads(repaired)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

    # Safe fallback for debugging.
    return {
        "raw_output": text,
        "parse_error": True,
    }


def format_json(data: Any) -> str:
    """Format Python objects as readable JSON text."""
    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
    )


def extract_final_chinese_summary(translation_result: Dict[str, Any]) -> str:
    """Extract the final Chinese summary from Agent 3's JSON output."""
    summary = translation_result.get("summary_zh", "")

    if isinstance(summary, list):
        return " ".join(str(item).strip() for item in summary if str(item).strip())

    if isinstance(summary, str):
        return summary.strip()

    if summary:
        return str(summary).strip()

    # Fallback if the model failed to return the expected JSON key.
    raw_output = translation_result.get("raw_output", "")

    if isinstance(raw_output, str):
        return raw_output.strip()

    return ""


def semantic_understanding_agent(dialogue: str) -> Dict[str, Any]:
    """Agent 1: Information Extraction Agent.

    Input:
        Original English dialogue

    Output:
        Structured representation of the dialogue
    """
    prompt = fill_prompt(
        SEMANTIC_UNDERSTANDING_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=SEMANTIC_UNDERSTANDING_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

    return extract_json_object(response)


def summary_planning_agent(
    semantic_representation: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 2: Summary Planning Agent.

    Input:
        Structured representation from Agent 1

    Output:
        Summary plan / summary points
    """
    prompt = fill_prompt(
        SUMMARY_GENERATION_PROMPT,
        {
            "semantic_representation": format_json(semantic_representation),
        },
    )

    response = call_ollama(
        model=SUMMARY_GENERATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return extract_json_object(response)


def translation_verification_agent(
    summary_plan: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 3: Translation-Verification Agent.

    Input:
        Summary plan from Agent 2

    Output:
        Final Chinese summary with verification information
    """
    summary_points = summary_plan.get("summary_points", summary_plan)

    prompt = fill_prompt(
        TRANSLATION_PROMPT,
        {
            "summary_points": format_json(summary_points),
        },
    )

    response = call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

    return extract_json_object(response)

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: Three-agent pipeline

def run_three_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the information extraction → summary planning → translation-verification pipeline."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: Information Extraction Agent
    structured_representation = semantic_understanding_agent(dialogue)

    # Agent 2: Summary Planning Agent
    summary_plan = summary_planning_agent(
        semantic_representation=structured_representation,
    )

    # Agent 3: Translation-Verification Agent
    translation_verification = translation_verification_agent(
        summary_plan=summary_plan,
    )

    final_chinese_summary = extract_final_chinese_summary(
        translation_result=translation_verification,
    )

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        "agent1_structured_representation": structured_representation,
        "agent1_structured_representation_json": format_json(structured_representation),

        "agent2_summary_plan": summary_plan,
        "agent2_summary_plan_json": format_json(summary_plan),

        "agent3_translation_verification": translation_verification,
        "agent3_translation_verification_json": format_json(translation_verification),

        "agent3_final_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [10]:
# Cell 9: Load top examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [11]:
# Cell 10: Run the three-agent pipeline for the first example from test.json

result = run_three_agent_pipeline(test_data[0])
result

{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_structured_representation': {'participants': ['Hannah', 'Amanda'],
  'semantic_grounding': [{'speaker': 'Hannah',
    'speech_act': 'Request for Information',
    'intended_meaning': "Hannah is asking Amanda if she possesses Betty's phone number.",
    'evidence': "Hey, do you have Betty's number?"},
   {'speaker': 'Amanda',
    'speech_act': 'Declarati

In [13]:
# Cell 11: Print three-agent pipeline result clearly

def print_three_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: Information Extraction / Structured Representation ===")
    print(result["agent1_structured_representation_json"])
    print()

    print("=== Agent 2: Summary Plan ===")
    print(result["agent2_summary_plan_json"])
    print()

    print("=== Agent 3: Translation + Verification ===")
    print(result["agent3_translation_verification_json"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])


print_three_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: Information Extraction / Structured Representation ===
{
  "participants": [
    "Hannah",
    "Amanda"
  ],
  "semantic_grounding": [
    {
      "speaker": "Hannah",
      "speech_act": "Request for Information",
      "intended_meaning": "Hannah is asking Amanda if she possesses Betty's phone number.",
      "evidence": "Hey, do you have Betty's number?"
    },
    {
      "speaker": "Amanda",
      "speech_act": "Declarative/Propositional Act",
      "intended_meaning": "Amanda states she is unable to locate the number in her contacts.",
    

## 4. Save Results

This saves all intermediate outputs and the final output.


In [14]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_outputs.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [15]:
# Cell 13: Batch inference with three-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running semantic-summary-translation pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_three_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running semantic-summary-translation pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_outputs.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [16]:
# Cell 14: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        "agent1_structured_representation_json": record.get(
            "agent1_structured_representation_json", ""
        ),
        "agent2_summary_plan_json": record.get(
            "agent2_summary_plan_json", ""
        ),
        "agent3_translation_verification_json": record.get(
            "agent3_translation_verification_json", ""
        ),
        "agent3_final_chinese_summary": record.get(
            "agent3_final_chinese_summary", ""
        ),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/1_semantic_summary_translation_final_summaries.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_structured_representation_json,agent2_summary_plan_json,agent3_translation_verification_json,agent3_final_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",汉娜不太愿意直接联系拉里，阿曼达主动提出帮汉娜发短信，表示愿意代为传达信息。 最终，汉娜接受...,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","{\n ""summary_points"": [\n ""Amanda agrees t...","{\n ""summary_zh"": [\n ""汉娜不太愿意直接联系拉里，阿曼达主动提...",汉娜不太愿意直接联系拉里，阿曼达主动提出帮汉娜发短信，表示愿意代为传达信息。 最终，汉娜接受...
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,罗证实了这位名为“Machine”的喜剧演员在YouTube上还有其他脱口秀视频，澄清了现在...,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","{\n ""summary_points"": [\n ""Rob confirmed t...","{\n ""summary_zh"": [\n ""罗证实了这位名为“Machine”的喜...",罗证实了这位名为“Machine”的喜剧演员在YouTube上还有其他脱口秀视频，澄清了现在...
2,test_00003,"Lenny: Babe, can you help me with something?\r...",鲍勃建议莱尼在选择裤子时，更应该注重质量，而不是拘泥于具体的颜色款式。他强调，选择裤子的主要...,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...","{\n ""summary_points"": [\n ""Bob advises Len...","{\n ""summary_zh"": [\n ""鲍勃建议莱尼在选择裤子时，更应该注重质...",鲍勃建议莱尼在选择裤子时，更应该注重质量，而不是拘泥于具体的颜色款式。他强调，选择裤子的主要...
3,test_00004,"Will: hey babe, what do you want for dinner to...",艾玛婉拒了威尔的好意，说自己现在不饿，希望他别费心准备晚餐。 艾玛也拒绝了威尔的提议，表示她...,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,"{\n ""participants"": [\n ""Will"",\n ""Emma...","{\n ""summary_points"": [\n ""Emma refuses Wi...","{\n ""summary_zh"": [\n ""艾玛婉拒了威尔的好意，说自己现在不饿，...",艾玛婉拒了威尔的好意，说自己现在不饿，希望他别费心准备晚餐。 艾玛也拒绝了威尔的提议，表示她...
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",简之前关于缺威士忌的说法只是开玩笑，并非真的需要，这消除了之前的误会。 简婉拒了明天的午餐邀...,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,"{\n ""participants"": [\n ""Ollie"",\n ""Jan...","{\n ""summary_points"": [\n ""Jane clarified ...","{\n ""summary_zh"": [\n ""简之前关于缺威士忌的说法只是开玩笑，并...",简之前关于缺威士忌的说法只是开玩笑，并非真的需要，这消除了之前的误会。 简婉拒了明天的午餐邀...


In [17]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,final_summary,reference_chinese_summary
0,test_00001,汉娜不太愿意直接联系拉里，阿曼达主动提出帮汉娜发短信，表示愿意代为传达信息。 最终，汉娜接受...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,test_00002,罗证实了这位名为“Machine”的喜剧演员在YouTube上还有其他脱口秀视频，澄清了现在...,埃里克和罗伯要在youtube上看一场单口相声。
2,test_00003,鲍勃建议莱尼在选择裤子时，更应该注重质量，而不是拘泥于具体的颜色款式。他强调，选择裤子的主要...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,test_00004,艾玛婉拒了威尔的好意，说自己现在不饿，希望他别费心准备晚餐。 艾玛也拒绝了威尔的提议，表示她...,艾玛很快就会回家，而且她会告诉威尔。
4,test_00005,简之前关于缺威士忌的说法只是开玩笑，并非真的需要，这消除了之前的误会。 简婉拒了明天的午餐邀...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [18]:
# Cell 16: Inspect intermediate outputs and final output

if not df.empty:
    inspection_columns = [
        "id",
        "agent1_structured_representation_json",
        "agent2_summary_plan_json",
        "agent3_translation_verification_json",
        "final_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,agent1_structured_representation_json,agent2_summary_plan_json,agent3_translation_verification_json,final_summary
0,test_00001,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","{\n ""summary_points"": [\n ""Amanda agrees t...","{\n ""summary_zh"": [\n ""汉娜不太愿意直接联系拉里，阿曼达主动提...",汉娜不太愿意直接联系拉里，阿曼达主动提出帮汉娜发短信，表示愿意代为传达信息。 最终，汉娜接受...
1,test_00002,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","{\n ""summary_points"": [\n ""Rob confirmed t...","{\n ""summary_zh"": [\n ""罗证实了这位名为“Machine”的喜...",罗证实了这位名为“Machine”的喜剧演员在YouTube上还有其他脱口秀视频，澄清了现在...
2,test_00003,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...","{\n ""summary_points"": [\n ""Bob advises Len...","{\n ""summary_zh"": [\n ""鲍勃建议莱尼在选择裤子时，更应该注重质...",鲍勃建议莱尼在选择裤子时，更应该注重质量，而不是拘泥于具体的颜色款式。他强调，选择裤子的主要...
3,test_00004,"{\n ""participants"": [\n ""Will"",\n ""Emma...","{\n ""summary_points"": [\n ""Emma refuses Wi...","{\n ""summary_zh"": [\n ""艾玛婉拒了威尔的好意，说自己现在不饿，...",艾玛婉拒了威尔的好意，说自己现在不饿，希望他别费心准备晚餐。 艾玛也拒绝了威尔的提议，表示她...
4,test_00005,"{\n ""participants"": [\n ""Ollie"",\n ""Jan...","{\n ""summary_points"": [\n ""Jane clarified ...","{\n ""summary_zh"": [\n ""简之前关于缺威士忌的说法只是开玩笑，并...",简之前关于缺威士忌的说法只是开玩笑，并非真的需要，这消除了之前的误会。 简婉拒了明天的午餐邀...
